In [ ]:
'''
=================================================
Milestone 3

Nama  : Raka Airlangga
Batch : CODA-020-RMT

Program ini dibuat untuk melakukan automatisasi transform dan load data penjualan kopi menggunakan Airflow. 
Data diekstrak dari sumber dataset yang kemudian ditransformasi menggunakan Pyspark, 
dan hasil transformasinya disimpan ke dalam Database MongoDB.

'''

In [ ]:
# Menggunakan library pandas untuk kebutuhan explorasi
import pandas as pd

In [ ]:
'''
Proses Load Data
'''
# Membaca file csv menggunakan syntax .read_csv
# Menyimpan file yang sudah dibaca tadi ke dalam variabel df
pd.read_csv('coffe_sales_raw.csv')
df = pd.read_csv('coffe_sales_raw.csv')

In [ ]:
# Memeriksa isi data dari file csv
df.head()

,hour_of_day,payment_type,total_price,coffee_name,day_part,day_name,month_name,number_day,number_month,date,time_details
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,2024-03-01,10:15:50
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14


In [ ]:
# Memeriksa struktur data untuk mendeteksi adanya anomali data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3547 entries, 0 to 3546
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   hour_of_day  3547 non-null   int64  
 1   cash_type    3547 non-null   str    
 2   money        3547 non-null   float64
 3   coffee_name  3547 non-null   str    
 4   Time_of_Day  3547 non-null   str    
 5   Weekday      3547 non-null   str    
 6   Month_name   3547 non-null   str    
 7   Weekdaysort  3547 non-null   int64  
 8   Monthsort    3547 non-null   int64  
 9   Date         3547 non-null   str    
 10  Time         3547 non-null   str    
dtypes: float64(1), int64(3), str(7)
memory usage: 304.9 KB


In [ ]:
'''
Tiap kolom memiliki jumlah baris yang sama, tidak ada missing value, dan tidak ada duplikasi. 
'''

In [5]:
'''
Mengubah tipe data kolom 'Date' menjadi datetime untuk tujuan validasi expectation 5.
'''
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')

In [6]:
'''
Mengubah tipe data kolom 'Time' menjadi datetime dan membuang  miliseconds agar terlihat lebih rapih
'''
df['Time'] = pd.to_datetime(df['Time'], format='mixed').dt.floor('s').dt.time

In [7]:
'''
Melakukan standarisasi nama kolom agar lebih mudah dibaca dan dipahami.
'''
df.rename(columns={'cash_type': 'payment_type', 'money':'total_price', 'Time_of_Day':'day_part', 'Weekday':'day_name', 'Month_name':'month_name', 'Weekdaysort':'number_day', 'Monthsort':'number_month', 'Date':'date', 'Time':'time_details'}, inplace=True)

In [ ]:
# Memeriksa apakah terdapat missing value pada tiap kolom nya
df.isnull().sum()

hour_of_day     0
payment_type    0
total_price     0
coffee_name     0
day_part        0
day_name        0
month_name      0
number_day      0
number_month    0
date            0
time_details    0
dtype: int64

In [ ]:
'''
Pemereiksaan ulang missing value pada setiap kolom. Hasilnya menunjukkan bahwa tidak ada missing value pada dataset ini.
'''

In [ ]:
# Memeriksa jika ada duplikasi
df.duplicated().sum()

np.int64(0)

In [11]:
'''
Pemereiksaan ulang duplikasi pada setiap kolom. Hasilnya menunjukkan bahwa tidak ada duplikasi pada dataset ini.
'''

'\nPemereiksaan ulang duplikasi pada setiap kolom. Hasilnya menunjukkan bahwa tidak ada duplikasi pada dataset ini.\n'

In [ ]:
# Memeriksa jumlah unique value pada tiap kolom
df.nunique()

hour_of_day       17
payment_type       1
total_price       13
coffee_name        8
day_part           3
day_name           7
month_name        12
number_day         7
number_month      12
date             381
time_details    3428
dtype: int64

In [ ]:
'''
- hour_of_day hanya mencatat 17 unique value yang mengindikasikan transaksi tidak dilakukan selama 24 jam non-stop
- hanya terdapat 1 metode pembayaran
- terdapat 8 menu kopi
- date memiliki 381 baris karena satu hari memiliki lebih dari satu penjualan 
- time_details berisi waktu spesifik transaksi terjadi
'''

In [ ]:
# Memeriksa statistik deskriptif sederhana
df.describe()

,hour_of_day,total_price,number_day,number_month,date
count,3547.000000,3547.000000,3547.000000,3547.000000,3547
mean,14.185791,31.645216,3.845785,6.453905,2024-10-04 17:34:43.676346
min,6.000000,18.120000,1.000000,1.000000,2024-03-01 00:00:00
25%,10.000000,27.920000,2.000000,3.000000,2024-07-17 12:00:00
50%,14.000000,32.820000,4.000000,7.000000,2024-10-10 00:00:00
75%,18.000000,35.760000,6.000000,10.000000,2025-01-11 00:00:00
max,22.000000,38.700000,7.000000,12.000000,2025-03-23 00:00:00
std,4.234010,4.877754,1.971501,3.500754,NaN


In [ ]:
'''
- hour_of_day memiliki nilai minimum 6 dan maksimum 22, yang menjawab temuan sebelumnya bahwa transaksi hanya dilakukan pada rentang waktu operasional (06.00 - 22.00).
- Selisih antara total_price terendah (18.2) dan tertinggi(38.7) adalah 20.5, yang menunjukkan variasi harga yang cukup besar di antara transaksi.
- harga min 18.12, tertinggi 38.70, mean 31.65, dan medain 32.82
- nilai medain sedikit lebih tinggi dari mean, sehingga distribusinya condong ke kiri
'''

In [ ]:
'''
Memeriksa setiap kolom untuk melihat unique value yang ada di dalamnya dan memastikan tidak ada nilai yang tidak valid (meminimalisir typo).
'''
#df['coffee_name'].unique()
#df['payment_type'].unique()
#df['day_part'].unique()
#df['day_name'].unique()
df['month_name'].unique()
#df['date'].unique()

<StringArray>
['Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan',
 'Feb']
Length: 12, dtype: str

In [ ]:
'''
Menyimpan hasil validasi ke dalam csv bersih dan disimpan dalam variabel df.
'''
import pandas as pd
df.to_csv('coffee_sales_clean.csv', index=False)
df = pd.read_csv('coffee_sales_clean.csv')

In [ ]:
'''
Membuat fact table dan dim table. 
Fact table akan berisi data transaksi kopi, sedangkan dim table akan berisi informasi tambahan terkait transaksi tersebut.
'''

In [22]:
# Dimensi Coffee > menghapus duplikasi nama kopi dan menambahkan kolom coffee_id sebagai primary key
# index +1 agar dimulai dari 1 bukan 0

dim_coffee = (
    df[['coffee_name']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Membuat kolom id menggunakan fungsi surrogate key
dim_coffee['coffee_id'] = dim_coffee.index +1

dim_coffee = dim_coffee[
    ['coffee_id', 'coffee_name']
]

dim_coffee.head(10)

,coffee_id,coffee_name
0,1,Latte
1,2,Hot Chocolate
2,3,Americano
3,4,Americano with Milk
4,5,Cocoa
5,6,Cortado
6,7,Espresso
7,8,Cappuccino


In [23]:
# Dimensi Date > menghapus duplikasi tanggal dan menambahkan kolom date_id sebagai primary key

dim_date = (
    df[['date', 'day_name', 'month_name', 'number_day', 'number_month']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_date['date_id'] = dim_date.index +1

In [24]:
# Menyusun ulang kolom pada dimensi date agar lebih rapih dan mudah dibaca

dim_date = dim_date[
    ['date_id',
     'date',
     'day_name',
     'number_day',
     'month_name',
     'number_month'
     ]
]
dim_date.head()

,date_id,date,day_name,number_day,month_name,number_month
0,1,2024-03-01,Fri,5,Mar,3
1,2,2024-03-02,Sat,6,Mar,3
2,3,2024-03-03,Sun,7,Mar,3
3,4,2024-03-04,Mon,1,Mar,3
4,5,2024-03-05,Tue,2,Mar,3


In [25]:
dim_time = (
    df[['time_details', 'hour_of_day','day_part']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_time['time_id'] = dim_time.index +1

dim_time = dim_time[
    ['time_id',
     'time_details',
     'hour_of_day',
     'day_part'
     ]
]
dim_time.head()

,time_id,time_details,hour_of_day,day_part
0,1,10:15:50,10,Morning
1,2,12:19:22,12,Afternoon
2,3,12:20:18,12,Afternoon
3,4,13:46:33,13,Afternoon
4,5,13:48:14,13,Afternoon


In [26]:
# Membuat salinan dataframe utama sebagai dasar untuk membuat fact table.

fact_sales = df.copy()

In [27]:
# Menggabungkan data transaksi dengan dimensi coffee untuk mendapatkan coffee_id

fact_sales = fact_sales.merge(
    dim_coffee[['coffee_id', 'coffee_name']], 
    on='coffee_name', how='left')

In [28]:
# Menggabungkan data transaksi dengan dimensi date untuk mendapatkan date_id

fact_sales = fact_sales.merge(
    dim_date[['date_id', 'date']], on='date', how='left'
)

In [29]:
# Menggabungkan data transaksi dengan dimensi time untuk mendapatkan time_id

fact_sales = fact_sales.merge(
    dim_time[['time_id', 'time_details']], 
    on='time_details', how='left')

In [30]:
# Memilih foreign key dan kolom yang relevan untuk fact table, serta menghapus kolom yang tidak diperlukan.

fact_sales = fact_sales[
    ['date_id',
     'time_id',
     'coffee_id',
     'payment_type',
     'total_price']
].copy()

In [31]:
# Membuat kolom sales_id sebagai primary key untuk fact table. Kolom ini akan berisi nilai unik yang dihasilkan dari range 1 hingga jumlah baris pada fact_sales.

fact_sales.insert(
    0,
    'sales_id',
    range(1, len(fact_sales) + 1)
)
fact_sales.head(10)

,sales_id,date_id,time_id,coffee_id,payment_type,total_price
0,1,1,1,1,card,38.7
1,2,1,2,2,card,38.7
2,3,1,3,2,card,38.7
3,4,1,4,3,card,28.9
4,5,1,5,1,card,38.7
5,6,1,6,4,card,33.8
6,7,1,7,2,card,38.7
7,8,1,8,4,card,33.8
8,9,1,9,5,card,38.7
9,10,1,10,4,card,33.8


In [1]:
'''
- Untuk menjalankan Great Expectations, kita perlu membuat Data Context terlebih dahulu. 
- Data Context adalah konfigurasi yang digunakan oleh Great Expectations untuk mengelola data dan ekspektasi.
- Hanya bisa dijalankan menggunakan kernel py310 sehingga perlu switch kernel terlebih dahulu.'''

from great_expectations.data_context import FileDataContext

context = FileDataContext.create(project_root_dir='./')

In [ ]:
#context.delete_datasource(datasource_name='fact_sales_datasource')

In [3]:
datasource_name = 'coffee_sales_datasource'
datasource = context.sources.add_pandas(datasource_name)

In [4]:
asset_name = 'coffee_sales_asset'
path_to_data = 'coffee_sales_clean.csv'
asset = datasource.add_csv_asset(asset_name, filepath_or_buffer=path_to_data)

In [5]:
batch_request = asset.build_batch_request()

In [6]:
'''
Membuat suite validasi data menggunakan Great Expectations untuk memastikan kualitas data tetap terjaga.
'''

expectation_suite_name = 'coffee_sales_suite'
context.add_or_update_expectation_suite(expectation_suite_name)

{
  "expectation_suite_name": "coffee_sales_suite",
  "ge_cloud_id": null,
  "expectations": [],
  "data_asset_type": null,
  "meta": {
    "great_expectations_version": "0.18.19"
  }
}

In [7]:
'''
Membuat validator untuk memeriksa kualitas data pada dataset coffee sales. Validator ini akan digunakan untuk menambahkan ekspektasi (expectations) yang sesuai dengan kebutuhan analisis data.
'''
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=expectation_suite_name
)
validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,hour_of_day,payment_type,total_price,coffee_name,day_part,day_name,month_name,number_day,number_month,date,time_details
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,2024-03-01,10:15:50
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14


In [ ]:
'''
EXPECTATIONS

Expectation 1: Memastikan kolom 'hour_of_day' memiliki nilai minimum 6 dan maksimum 22.
'''

validator.expect_column_values_to_be_between(
    column='hour_of_day', min_value=6, max_value=22
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3547,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

Validasi sukses

In [9]:
'''
Expectation 2: Memastikan kolom 'payment_type' hanya memiliki satu nilai unik, yaitu 'card'.
'''

validator.expect_column_distinct_values_to_be_in_set(
    column='payment_type', value_set=['card']
)


Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": [
      "card"
    ],
    "details": {
      "value_counts": [
        {
          "value": "card",
          "count": 3547
        }
      ]
    }
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

Validasi Sukses

In [10]:
'''
Expectation 3: Memastikan kolom 'coffee_name' tidak bernilai null.
'''
validator.expect_column_values_to_not_be_null(column='coffee_name')

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3547,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

Validasi Sukses

In [11]:
'''
Expectation 4: Memastikan kolom 'total_price' memiliki nilai > dari 0.
'''
validator.expect_column_values_to_be_between(
    column='total_price', min_value=0, strict_min=True
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3547,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

Validasi Sukses

In [12]:
'''
Expectations 5: Memastikan kolom 'date' memiliki format tanggal yang valid.
'''

validator.expect_column_values_to_be_dateutil_parseable(
    column='date'
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3547,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

Validasi Sukses

In [15]:
context.list_datasources()

[{'type': 'pandas',
  'name': 'fact_sales_datasource',
  'assets': [{'name': 'fact_sales_asset',
    'type': 'dataframe',
    'batch_metadata': {}}]},
 {'type': 'pandas',
  'name': 'coffee_sales_datasource',
  'assets': [{'name': 'coffee_sales_asset',
    'type': 'csv',
    'filepath_or_buffer': WindowsPath('coffee_sales_clean.csv')}]}]

In [18]:
# Datasource untuk fact_sales

fact_datasource_name = 'fact_sales_datasource'

fact_datasource = context.sources.add_pandas(fact_datasource_name)

In [19]:
# Dataframe asset dari fact_sales

fact_asset = fact_datasource.add_dataframe_asset(
    name='fact_sales_asset'
)

In [32]:
# Build batch request untuk fact_sales

fact_batch_request = fact_asset.build_batch_request(
    dataframe=fact_sales
)

In [33]:
# Suite validasi untuk fact_sales

fact_suite_name = 'fact_sales_suite'
context.add_or_update_expectation_suite(fact_suite_name)

{
  "expectation_suite_name": "fact_sales_suite",
  "ge_cloud_id": null,
  "expectations": [],
  "data_asset_type": null,
  "meta": {
    "great_expectations_version": "0.18.19"
  }
}

In [34]:
# Validator untuk fact_sales

fact_validator = context.get_validator(
    batch_request=fact_batch_request,
    expectation_suite_name=fact_suite_name
)

In [35]:
fact_validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,sales_id,date_id,time_id,coffee_id,payment_type,total_price
0,1,1,1,1,card,38.7
1,2,1,2,2,card,38.7
2,3,1,3,2,card,38.7
3,4,1,4,3,card,28.9
4,5,1,5,1,card,38.7


In [36]:
'''
Expectation 6: Memastikan kolom 'sales_id' memiliki nilai unik
'''

fact_validator.expect_column_values_to_be_unique(column='sales_id')

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 3547,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [37]:
'''
Expectation 7: Memastikan kolom 'id' masuk ke dataframe fact_sales.
'''

fact_validator.expect_column_to_exist(column='sales_id')
fact_validator.expect_column_to_exist(column='date_id')
fact_validator.expect_column_to_exist(column='time_id')
fact_validator.expect_column_to_exist(column='coffee_id')

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

{
  "success": true,
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [38]:
'''Menyimpan Validator ke dalam expectation suite'''

validator.save_expectation_suite(discard_failed_expectations=False)
fact_validator.save_expectation_suite(discard_failed_expectations=False)

In [39]:
'''Membuat chechkpoint untuk memvalidasi data menggunakan expectation suite yang telah dibuat sebelumnya. Checkpoint ini akan digunakan untuk menjalankan validasi data secara otomatis'''

checkpoint_1 = context.add_or_update_checkpoint(
    name='checkpoint_1',
    validator = validator,
)

# Cara menjalankan checkpoint
checkpoint_result = checkpoint_1.run()

Calculating Metrics:   0%|          | 0/30 [00:00<?, ?it/s]

In [40]:
# Checkpoint untuk fact sales

checkpoint_fact = context.add_or_update_checkpoint(
    name='checkpoint_fact',
    validator=fact_validator
)

checkpoint_fact_result = checkpoint_fact.run()

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

In [41]:

context.build_data_docs()
# Output ada di folder gx > uncomited > data_docs > index.html > buka di browser

{'local_site': 'file://c:\\TMP\\Phase 2\\milestone_3\\gx\\uncommitted/data_docs/local_site/index.html'}